# BERT news classification

**Manahil Iftikhar · DevelopersHub internship portfolio**

Transformer training, text classification, and evaluation

**Execution status:** Training code present; completed evaluation not in the archive

## Data and evidence

AG News via Hugging Face Datasets; bert-base-uncased. The original selected 10,000 training and 2,000 test examples.

The saved training progress stops at 3/1,875 steps. No completed accuracy or F1 result is recorded, so the earlier README’s approximately 92% accuracy claim is not retained.

## Maintained workflow

The maintained notebook reserves validation data from the training pool, keeps test data out of checkpoint selection, handles inference device placement, and makes the demo launch opt-in.

Original assignment title: *News Topic Classifier Using BERT*. Original code and outputs are preserved in `archive/`.

Install the environment described in the repository README, then select it as the VS Code notebook kernel. Outputs below are intentionally cleared until this maintained version is executed.

In [ ]:
from pathlib import Path
import sys

# Supports opening the notebook from the repository root or notebooks/ in VS Code.
ROOT = Path.cwd()
if not (ROOT / 'portfolio').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'portfolio').is_dir():
    raise RuntimeError('Open this notebook inside the cloned repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)

In [ ]:
# News Topic Classifier Using BERT - Complete Google Colab Implementation
# AI/ML Engineering Internship Task 1 - DevelopersHub Corporation

## Step 1: Install Required Libraries

## Step 2: Import Required Libraries and Disable WandB

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  # Completely disable WandB

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import evaluate
import gradio as gr
from torch.nn.functional import softmax

## Step 3: Load and Explore Dataset

In [ ]:
print("Loading AG News Dataset...")
dataset = load_dataset("ag_news")

# Display dataset info
print("Dataset Structure:")
print(dataset)
print("\nDataset Features:")
print(dataset['train'].features)
print("\nSample Data:")
print(dataset['train'][:5])

# Label mapping
label_names = ['World', 'Sports', 'Business', 'Science/Technology']
print(f"\nLabel Categories: {label_names}")

## Step 4: Initialize Tokenizer and Model

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

print(f"Model loaded: {model_name}")
print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")

## Step 5: Tokenization Function

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=512
    )

## Step 6: Tokenize Dataset

In [ ]:
print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Create smaller dataset for faster training (optional - remove if you want full dataset)
training_pool = tokenized_datasets['train'].shuffle(seed=42).select(range(10000))
split = training_pool.train_test_split(test_size=0.2, seed=42)
train_dataset = split['train']
validation_dataset = split['test']
test_dataset = tokenized_datasets['test'].shuffle(seed=42).select(range(2000))

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## Step 7: Data Collator

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Step 8: Evaluation Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')

    return {
        'accuracy': accuracy['accuracy'],
        'f1': f1['f1']
    }

## Step 9: Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",  # Fixed: changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to=[]  # Disable wandb logging
)

## Step 10: Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Step 11: Train the Model

In [ ]:
print("Starting training...")
trainer.train()

## Step 12: Evaluate the Model

In [ ]:
print("Evaluating model...")
eval_results = trainer.evaluate(eval_dataset=test_dataset)
print("Evaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

## Step 13: Make Predictions on Test Set

In [ ]:
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

# Detailed evaluation
print("\nDetailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_names))

## Step 14: Save Model

In [ ]:
model_save_path = "./news_classifier_bert"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to: {model_save_path}")

## Step 15: Prediction Function

In [ ]:
def predict_news_category(text):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = softmax(outputs.logits, dim=-1)

    # Get predicted class and confidence
    predicted_class_id = predictions.argmax().item()
    confidence = predictions[0][predicted_class_id].item()

    return label_names[predicted_class_id], confidence

## Step 16: Test Individual Predictions

In [ ]:
print("\n" + "="*50)
print("TESTING INDIVIDUAL PREDICTIONS")
print("="*50)

test_headlines = [
    "Apple stock rises 5% after strong quarterly earnings",
    "Scientists discover new species of deep-sea fish",
    "Manchester United defeats Chelsea 3-1 in Premier League",
    "UN Security Council meets to discuss global climate crisis"
]

for headline in test_headlines:
    category, confidence = predict_news_category(headline)
    print(f"\nHeadline: {headline}")
    print(f"Predicted Category: {category}")
    print(f"Confidence: {confidence:.4f}")

## Step 17: Gradio Interface for Deployment

In [ ]:
def gradio_predict(text):
    if not text.strip():
        return "Please enter a news headline!", 0.0

    try:
        category, confidence = predict_news_category(text)
        return f"Category: {category}", f"Confidence: {confidence:.4f}"
    except Exception as e:
        return f"Error: {str(e)}", "N/A"

# Create Gradio interface
iface = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Enter a news headline here...",
        label="News Headline"
    ),
    outputs=[
        gr.Textbox(label="Predicted Category"),
        gr.Textbox(label="Confidence Score")
    ],
    title="🗞️ News Topic Classifier Using BERT",
    description="Enter a news headline and get its predicted category (World, Sports, Business, or Science/Technology)",
    examples=[
        ["Apple announces new iPhone with revolutionary camera technology"],
        ["FIFA World Cup final draws record television audience"],
        ["Stock markets rally after Federal Reserve announcement"],
        ["NASA launches new Mars exploration mission"]
    ],
    theme="default"
)

## Step 18: Launch Gradio Interface

In [ ]:
print("\n" + "="*50)
print("LAUNCHING GRADIO INTERFACE")
print("="*50)
print("The interface will open in a new tab/window")

# Launch with public sharing for Colab
RUN_LOCAL_DEMO = False
if RUN_LOCAL_DEMO:
    iface.launch(share=False)

## Interpretation

The selected checkpoint is evaluated on the held-out test subset only after training. Report the actual resulting scores and hardware/runtime. No trained checkpoint or verified test score is bundled. A temporary interface is not a hosted deployment.